# 문화누리카드 가맹점 마스터 테이블 생성

목적: 데이터 전처리 및 필요 attribute 결정, 그리고 핵심 정보 EDA
- 문화누리카드 가맹점 원자료의 위치, 분류, 이용서비스 정보를 정제하여 도보 및 대중교통 접근성 분석에 활용 가능한 가맹점 단위 정보를 생성한다.
- 필요정보: 가맹점 ID, 분류, 위치정보, 기타 편의시설(장애인, 노인)과 시군구·행정동 구역 정보
* 접근성을 계산할 수 있는 정보들이 필수로 담겨야 함


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pathlib
import numpy as np
import pathlib
from matplotlib.colors import LinearSegmentedColormap
import mapclassify as mc

salmon_cmap = LinearSegmentedColormap.from_list(
    "salmon_cmap",
    ["#fff5f0", "#fddbc7", "#f4a582", "#ef8a62", "#d6604d", "#b2182b"]
)

plt.rcParams["font.family"] = "Noto Sans KR"

BASE_PATH = pathlib.Path().resolve()
if BASE_PATH.name == 'notebooks':
    BASE_PATH = BASE_PATH.parent.parent

if BASE_PATH.name == 'analysis_table':
    BASE_PATH == BASE_PATH.parent

ANALYSIS_PATH = BASE_PATH / "analysis_table"
ANALYSIS_PATH.mkdir(parents = True, exist_ok = True)

DATA_PATH = ANALYSIS_PATH / "data"
DATA_PATH.mkdir(parents = True, exist_ok = True)

INPUT_PATH = DATA_PATH / "input"
INPUT_PATH.mkdir(parents = True, exist_ok=True)

OUTPUT_PATH = DATA_PATH / "output"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

IMAGE_PATH = ANALYSIS_PATH / "image"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

RAW_PATH = BASE_PATH / "data" / "raw"

SPATIAL_PATH = RAW_PATH / "spatial"

## 데이터 불러오기 및 데이터 칼럼 정제

    # 첫 번째 행에 대분류·중분류·소분류·시군구 정보가 보조 헤더 형태 
    # 해당 행은 실제 가맹점 레코드가 아니므로 제거
    # 이후 가맹점명, 대분류, 중분류, 소분류, 위도, 경도, 시군구, 주소, 이용정보, 연관검색어, URL, 이용분류 관련 칼럼 선택
    #  `Unnamed` 칼럼과 줄바꿈이 포함된 칼럼명을 분석에 용이한 칼럼명으로 변환

In [ ]:
store = pd.read_excel(RAW_PATH / "merchants" / "source" / "mnc_seoul_offline_merchants_20260706.xlsx")

# 데이터 품질 점검 및 필요 칼럼 정리
print(store.columns)
print(store.shape)
print(store.isna().sum())


display(store.loc[0, :])
display(store.head())
display(store["지역"].value_counts())

# 1행의 일부 정보가 보조 헤더 역할
# 분야: 대분류, 지역: 시 , 'Unnamed: 4': 중분류, 'Unnamed: 5: 소분류, Unnamed: 12: 시군구

# 1행 제거
store = store.iloc[1:].copy()
store = store.reset_index(drop=True)

# 필요 칼럼 선택
store = store[['가맹점명', '분야', 'Unnamed: 4', 'Unnamed: 5', '위도', '경도',
       '이용정보', 'Unnamed: 12', '주소',
       '연관검색어', 'URL',  '이용분류', '이용분류\n전화결제','이용분류\n상세내용']].copy()


# 칼럼명 정리
store.rename(columns = {"분야": '대분류', 
                        'Unnamed: 4': '중분류', 
                        'Unnamed: 5': '소분류', 
                        'Unnamed: 12': '시군구',
                        '이용분류\n전화결제': '이용분류_전화결제',
                        '이용분류\n상세내용': '이용분류_상세내용'},
             inplace = True,
             errors = 'ignore')

print(f"칼럼 선택 결과 확인: {store.columns}")
    # 칼럼 선택 결과 확인: Index(['가맹점명', '대분류', '중분류', '소분류', '위도', '경도', '이용정보', '시군구', '주소', '연관검색어',
    #        'URL', '이용분류', '이용분류_전화결제', '이용분류_상세내용'],
    #       dtype='str')


## 데이터 전처리
    # 결측치 처리
        # 가맹점 데이터의 결측 현황과 서비스 관련 칼럼의 내용을 확인함. 
        # 위치와 분류 관련 핵심 칼럼은 결측이 없어 접근성 분석에 활용 가능함을 확인했으며, 
        # 이용분류 칼럼은 전화결제, 찾아가는 문화서비스, 장애인친화시설 여부를 구분하는 이진 변수 생성에 활용하기로 함. 
        # 반면 이용분류_전화결제는 대부분 결측이고 전화번호 자체가 접근성 분석의 핵심 변수가 아니므로 제외함.
        #  이용정보와 이용분류_상세내용은 향후 키워드 분석 또는 외부 데이터 보완을 통해 부가적인 서비스 특성 변수로 활용할 수 있음.

    # 중복값 처리
        # 중복처리 기준: 가맹점명-중분류-소분류 기준 중복 (동일한 문화서비스를 제공하는 동일한 이름의 시설)
        # 미터계 좌표 기준 5m 이내는 동일한 시설이라고 판단
        # 정밀한 비교 위해 미터계 좌표(EPSG:5179)로 변환
        # 가맹점명-중분류-소분류 중복 레코드에 대해 geometry 거리 계산 후 5m 미만의 시설을 중복 시설이라고 정의
        # 총 5개의 중복행 제거

    # 중복 레코드
        #       가맹점명	중분류	소분류	거리m	idx_1	idx_2
        # 1	    극단친구	공연	공연	0.000000	3709	3714
        # 8	    면목제4동 주민센터	문화체험	문화체험	0	1647	1711
        # 12	미래서적	도서	도서	0.000000	307	540
        # 13	베토벤	    음악	음악	0.052191	2605	4383
        # 15	샐몬모텔	숙박	숙박	0.100241	195	4322




In [ ]:
# 결측치 확인
    #     가맹점명	중분류	소분류	거리m	idx_1	idx_2
    # 1	극단친구	공연	공연	0.000000	3709	3714
    # 8	면목제4동 주민센터	문화체험	문화체험	0.000000	1647	1711
    # 12	미래서적	도서	도서	0.000000	307	540
    # 13	베토벤	음악	음악	0.052191	2605	4383
    # 15	샐몬모텔	숙박	숙박	0.100241	195	4322
        # 가맹점 데이터의 결측 현황과 서비스 관련 칼럼의 내용을 확인함. 
        # 위치와 분류 관련 핵심 칼럼은 결측이 없어 접근성 분석에 활용 가능함을 확인했으며, 
        # 이용분류 칼럼은 전화결제, 찾아가는 문화서비스, 장애인친화시설 여부를 구분하는 이진 변수 생성에 활용하기로 함. 
        # 반면 이용분류_전화결제는 대부분 결측이고 전화번호 자체가 접근성 분석의 핵심 변수가 아니므로 제외함.
        #  이용정보와 이용분류_상세내용은 향후 키워드 분석 또는 외부 데이터 보완을 통해 부가적인 서비스 특성 변수로 활용할 수 있음.
    # 가맹점명            0
    # 대분류             0
    # 중분류             0
    # 소분류             0
    # 위도              0
    # 경도              0
    # 이용정보         1582
    # 시군구             0
    # 주소              0
    # 연관검색어           0
    # URL          3772
    # 이용분류         4282
    # 이용분류_전화결제    4486
    # 이용분류_상세내용    4459
    # dtype: int64

print(store.shape)
print(store.isna().sum())
print(store["이용분류"].unique()) # 전화결제, 장애인친화시설, 찾아가는 문화서비스 등 서비스 여부 - 포함항목 별로 나눠서 새로운 칼럼 유형 생성, 추가 웹크롤링으로 데이터 보완 가능
print(store["이용분류_전화결제"].unique()) # 전화번호 - 필요없음
print(store["이용정보"].unique()) # 시설에 관한 정보 - 키워드 분석으로 사용할 수는 있을듯? 결측치는 네이버 크롤링으로 커버 가능할수도
print(store["이용분류_상세내용"].unique()) # 전화결제, 장애인친화시설, 찾아가는 문화서비스 등 서비스 여부에 관한 상세 설명 - 키워드 분석으로 사용할 수 있을듯? 어떻게 사용할지는 아직 미정

In [ ]:
# 이용분류_전화결제 칼럼 삭제
store = store.drop(columns = "이용분류_전화결제",
                   errors = 'ignore')


# 이용분류 - 서비스 별 칼럼 생성
store['전화결제'] = store["이용분류"].astype(str).str.contains('전화결제').astype(int)
store['장애인친화시설'] = store["이용분류"].astype(str).str.contains('장애인친화시설').astype(int)
store['찾아가는문화서비스'] = store["이용분류"].astype(str).str.contains('찾아가는문화서비스').astype(int)
store = store.drop(columns = '이용분류',
                   errors = 'ignore')

# 칼럼 정리
store = store [['가맹점명', '시군구', '주소', '위도', '경도', '대분류', '중분류', '소분류', '전화결제', '장애인친화시설', '찾아가는문화서비스', '이용정보', '연관검색어',
       '이용분류_상세내용', 'URL']]

In [ ]:
# 위도 0.00001도 ≈ 약 1.1m
# 경도 0.00001도 ≈ 서울 기준 약 0.9m
# 서울시 좌표는 4326보다 5179로 좌표변환후 좌표 확인하는 것이 더 중요.

# 서울시 기준 데카르트 좌표계로 변환
store = store.copy()
store = gpd.GeoDataFrame(store,
                         geometry = gpd.points_from_xy(store["경도"], store["위도"]),
                         crs = "EPSG: 4326")
store = store.to_crs("EPSG: 5179")


# 중복 제거 기준 : 가맹점 명 - 중분류, 소분류 동일, geometry 거리가 몇 미터 이내
    # 가맹점명_중분류_소분류 중복 후보 행 개수: 87
store_dup = store[["가맹점명", "중분류", "소분류"]].duplicated(keep=False)

dup_store = store[store_dup].copy()
print(f"가맹점명_중분류_소분류 중복 후보 행 개수: {len(dup_store)}")
display(dup_store.sort_values(["가맹점명", "중분류", "소분류"]))

# 같은 가맹점명 + 중분류 + 소분류 조합 geometry 비교
dup_dist_list = []
for key, group in dup_store.groupby(["가맹점명", "중분류", "소분류"]): # key는 그룹 키값, group은 그 키에 해당하는 행들
    group = group.copy()
    for i in range(len(group)):
        for j in range(i+1, len(group)):
            idx_i = group.index[i]
            idx_j = group.index[j]
            
            dist_m = group.loc[idx_i, "geometry"].distance(group.loc[idx_j, "geometry"])
            
            dup_dist_list.append({"가맹점명": key[0],
                                  "중분류": key[1],
                                  "소분류": key[2],
                                  "거리m": dist_m,
                                  "idx_1": idx_i,
                                  "idx_2": idx_j})

dup_store_table = pd.DataFrame(dup_dist_list)
display(dup_store_table.sort_values("거리m", ascending=True))

dup_idx = dup_store_table.loc[(dup_store_table["거리m"] < 5), "idx_2"].unique()
print(dup_idx)

# 기준 중복 행 제거
store_clean= store.drop(index = dup_idx)
print(f"중복 행 제거 개수: {len(store) - len(store_clean)}")


## Point In Polygon 분석 기반 가맹점 위치 POI 유효성 검증
    # EPSG:4326으로 변환 후 분석 진행
    # PIP 분석 결과 총 20개의 가맹점이 서울 경계 바깥으로 확인
    # 해당 레코드 삭제 처리
    # 최종 가맹점 개수 : 4702개

In [ ]:
# 서울시 행정동 경계 불러오기
bound_kr = gpd.read_file(SPATIAL_PATH / "boundary" / "BND_ADM_DONG_PG.shp")

bound = bound_kr[bound_kr["ADM_CD"].astype(str).str.startswith("11")]

print(f"crs 확인: {bound.crs}")
print(f"지오타입 확인: {bound.geometry.geom_type.unique()}")
print(f"데이터 구조 확인: {bound.shape}")
bound.plot()
display(bound.head())

# 서울시 행정동 경계 칼럼 정리
bound = bound.copy()
bound = bound[["ADM_NM", "geometry"]]
bound = bound.rename(columns = {"ADM_NM": "행정동"})


# 시군구 행정동 데이터 불러오기
hjd_code = pd.read_excel(
    SPATIAL_PATH / "boundary" / "BND_ADM_DONG_PG_geocode.xlsx",
    sheet_name=0,
    header=1
)

# 시군구 행정동 데이터 칼럼 정리
# 서울시 행정동 경계 불러오기
bound_kr = gpd.read_file(
    SPATIAL_PATH / "boundary" / "BND_ADM_DONG_PG.shp",
    encoding="cp949"
)

bound_kr["ADM_CD"] = bound_kr["ADM_CD"].astype(str)

# 서울시 행정동만 선택
bound = bound_kr[bound_kr["ADM_CD"].str.startswith("11")].copy()

print(f"crs 확인: {bound.crs}")
print(f"지오타입 확인: {bound.geometry.geom_type.unique()}")
print(f"데이터 구조 확인: {bound.shape}")

# 시군구 행정동 코드표 불러오기
hjd_code = pd.read_excel(
    SPATIAL_PATH / "boundary" / "BND_ADM_DONG_PG_geocode.xlsx",
    sheet_name=0,
    header=1
)

# 코드 컬럼 정리
hjd_code = hjd_code[["시도코드", "시군구코드", "시군구명칭", "읍면동코드", "읍면동명칭"]].copy()

hjd_code["시도코드"] = hjd_code["시도코드"].astype(str).str.zfill(2)
hjd_code["시군구코드"] = hjd_code["시군구코드"].astype(str).str.zfill(3)
hjd_code["읍면동코드"] = hjd_code["읍면동코드"].astype(str).str.zfill(3)

hjd_code["ADM_CD"] = (
    hjd_code["시도코드"]
    + hjd_code["시군구코드"]
    + hjd_code["읍면동코드"]
)

# 서울시 코드만 선택
hjd_code = hjd_code[hjd_code["시도코드"] == "11"].copy()

hjd_code = hjd_code[["ADM_CD", "시군구명칭", "읍면동명칭"]].rename(
    columns={
        "시군구명칭": "시군구",
        "읍면동명칭": "행정동"
    }
)

# 경계 + 시군구/행정동명 결합
bound = bound.merge(
    hjd_code,
    on="ADM_CD",
    how="left"
)

bound = bound[["ADM_CD", "시군구", "행정동", "geometry"]].copy()

print(f"결합 후 구조: {bound.shape}")
print(f"결합 후 결측:\n{bound.isna().sum()}")
print(f"결합 후 타입: {type(bound)}")

display(bound.head())

# 테이블 저장
bound.to_file(OUTPUT_PATH / "서울시_시군구_행정동_경계.gpkg",
              driver = "GPKG")

In [ ]:
# 행정동을 외곽경계로
seoul_bound = bound.dissolve()
display(seoul_bound)
print(seoul_bound.crs)
seoul_bound.plot(facecolor = 'None',
                 edgecolor = 'black',
                 linewidth = 1.2)

# crs 변경
seoul_bound = seoul_bound.to_crs(store_clean.crs)
print(f"변경 crs: {seoul_bound.crs}")

store_seoul = gpd.sjoin(store_clean,
                        seoul_bound[["geometry"]],
                        how =  'left',
                        predicate = 'within')

store_out = store_seoul[store_seoul["index_right"].isna()]

print("\n서울 경계 바깥 가맹점")
display(store_out)
print(f"경게 바깥 가맹점 개수: {len(store_out)}")

# 경게 바깥 가맹점 제거 및 불필요 칼럼 삭제
store_seoul = store_seoul[~(store_seoul["index_right"].isna())].copy()
store_seoul = store_seoul.drop(columns = 'index_right')

# 가맹점 위치 확인
fig, ax = plt.subplots(figsize=(8, 8))
seoul_bound.plot(ax=ax,
                 edgecolor = 'black',
                 alpha = 0.7,
                 facecolor = "None")
store_seoul.plot(ax=ax,
                 color = 'salmon',
                 marker = '+')
plt.title("서울시 문화누리 가맹점 분포")
ax.set_axis_off()
plt.savefig(IMAGE_PATH / "서울시_문화누리가맹점_위치_분포.png",
            bbox_inches = 'tight',
            pad_inches = 0.1,
            dpi = 240)
plt.show()

print(f"최종 가맹점 개수: {len(store_seoul)}")

In [ ]:
store_seoul = store_seoul.copy()

store_seoul = store_seoul.sort_values(
    ["시군구", "가맹점명", "중분류", "소분류", "주소"]
).reset_index(drop=True)

store_seoul["가맹점_ID"] = [
    f"STORE_{i:05d}" for i in range(1, len(store_seoul) + 1)
]

store_seoul = store_seoul[
    [
        "가맹점_ID",
        "가맹점명",
        "시군구",
        "주소",
        "위도",
        "경도",
        "대분류",
        "중분류",
        "소분류",
        "전화결제",
        "장애인친화시설",
        "찾아가는문화서비스",
        "이용정보",
        "연관검색어",
        "이용분류_상세내용",
        "URL",
        "geometry"
    ]
]

bound_seoul = gpd.read_file(OUTPUT_PATH / "서울시_시군구_행정동_경계.gpkg")

store_seoul_hjd = gpd.sjoin(store_seoul,
                        bound_seoul[["행정동", "geometry"]],
                        how = 'left',
                        predicate = 'within')

print(store_seoul_hjd["행정동"].isna().sum())
print(len(store_seoul), len(store_seoul_hjd))

## 가맹점-행정동 결합 : PIP 방식
가맹점 제공 geometry Points와 행정동 경계 Polygon을 PIP 기반(within 메서드)로 공간 결합 

In [ ]:
bound_seoul = gpd.read_file(OUTPUT_PATH / "서울시_시군구_행정동_경계.gpkg")

print(bound_seoul.crs)

bound_seoul = bound_seoul.to_crs(store_seoul.crs)

print(f"변경 후 crs: {bound_seoul.crs}")

store_seoul_hjd = gpd.sjoin(store_seoul,
                        bound_seoul[["행정동", "geometry"]],
                        how = 'left',
                        predicate = 'within').drop(columns = 'index_right',
                                                   errors = 'ignore')

print(store_seoul_hjd["행정동"].isna().sum())
print(len(store_seoul), len(store_seoul_hjd))
display(store_seoul_hjd.head(10))

# 칼럼 선택
store_seoul_final = store_seoul_hjd[
    [
        "가맹점_ID",
        "가맹점명",
        "시군구",
        "행정동",
        "주소",
        "위도",
        "경도",
        "대분류",
        "중분류",
        "소분류",
        "전화결제",
        "장애인친화시설",
        "찾아가는문화서비스",
        "이용정보",
        "연관검색어",
        "이용분류_상세내용",
        "URL",
        "geometry"
    ]
]

store_seoul_final.head()


## 문화누리카드 가맹점 마스터 테이블(최종) 저장

In [ ]:
# 간단 EDA
store_eda = store_seoul.copy()

store_type_mid = store_eda.groupby(by= "중분류").size()
store_type_small = store_eda.groupby(by= "소분류").size()
store_size_gu = store_eda.groupby(by = "시군구").size()

print(store_type_mid.sort_values(ascending=False))
print(store_type_small.sort_values(ascending=False))
print(store_size_gu.sort_values(ascending=False))

# 테이블 저장
store_seoul.to_file(OUTPUT_PATH / "서울시_문화누리카드_가맹점_2026.gpkg",
                    driver = "GPKG")


In [ ]:
a = gpd.read_file(OUTPUT_PATH / "서울시_문화누리카드_가맹점_2026.gpkg")
a.head()

a[a["중분류"] == '음악'].head(30)